# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a complete example for loading and exploring the [FAIR^2 dataset Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors](https://doi.org/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema at

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` and plotting libraries are installed
!pip install mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This will allow us to review available record sets, extract table data, and explore record field types.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # Metadata as object

print("Dataset Title:", metadata.name)
print("Description:", metadata.description)
print("Version:", getattr(metadata, 'version', 'N/A'))

## 2. Data Overview
Let's review available record sets, fields, and their `@id`s as defined by the Croissant schema. All references to entities (record sets, fields, columns) should use their precise `@id`.

**Note:** Because Croissant datasets can describe more than one record set and each field and column is precisely identified, this information guides reliable data access.

In [ ]:
# List available record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields:")
        for f in fields:
            # When fields are links, extract @id
            if isinstance(f, dict) and '@id' in f:
                print(f"    - {f['@id']}")
            elif isinstance(f, str):
                print(f"    - {f}")
            else:
                print(f"    - {f}")
    else:
        print("  (No fields found)")

## 3. Data Extraction
Now let's extract data for each record set. We'll load these into pandas DataFrames for further examination. All record sets and field selections use exact `@id` references for reproducibility.

**Tip:** After running the previous overview, choose the specific record set `@id` you want to analyze (there is typically one main tabular record set; if in doubt pick the primary table from the printout above).


In [ ]:
# Collect all record set @id's for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# Download each record set and store as DataFrame
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id} with shape {df.shape}")
    except Exception as e:
        print(f"Could not load data for {record_set_id}: {e}")

# Show columns for the main record set (if only 1, use it; if several, pick the largest)
if len(dataframes) > 0:
    # Choose the first or the one with more than 0 columns
    main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[1] * dataframes[k].shape[0])
    print(f"\nColumns in main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No data frames loaded.")

## 4. Exploratory Data Analysis (EDA)
Next, let's conduct some basic explorations with the main record set. We'll use only the field `@id`s for all references.

Common steps:
- Filter the data based on a numeric variable.
- Normalize a numeric variable.
- Group and summarize by a key attribute.

Please update the selected field `@id`s in the code below if needed after inspecting your data.

In [ ]:
# EDA on the main record set
# Replace with real @id from the output above
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Attempt to auto-detect a numeric field by inspecting dtypes
numeric_fields = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
if len(numeric_fields) == 0:
    # Try object columns that may be numbers
    for col in df.columns:
        try:
            pd.to_numeric(df[col].dropna().iloc[0])
            numeric_fields.append(col)
        except Exception:
            continue

if len(numeric_fields) == 0:
    print("No numeric fields auto-detected. Please set 'numeric_field_id' manually.")
    numeric_field_id = None
else:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")

# Filter by a threshold; set threshold as appropriate
threshold = 10
if numeric_field_id is not None:
    numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[numeric_series > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize values
    filtered_df[f"{numeric_field_id}_normalized"] = (numeric_series[numeric_series > threshold] - numeric_series[numeric_series > threshold].mean()) / numeric_series[numeric_series > threshold].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt to auto-detect a nominal/categorical column for grouping
    group_candidates = [c for c in df.columns if df[c].dtype == object and c != numeric_field_id]
    group_field_id = group_candidates[0] if group_candidates else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable field found for grouping.")
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field, as well as its grouping by the categorical field if present and filtered.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and not filtered_df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id].astype(float), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
    plt.show()

    # If grouped_df exists, plot bar plot
    if 'grouped_df' in locals() and group_field_id is not None:
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} grouped by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load and explore a multi-field clinical dataset using `mlcroissant` with precise Croissant `@id` referencing for record sets and fields.
- Inspect available record sets and fields, extract full tabular data, and conduct example exploratory data analysis (EDA), including filtering, normalization, and summary statistics.
- Visualize data distributions and trends.

This workflow is adaptable to any dataset described with the Croissant standard and the `mlcroissant` toolkit.